# RHO-FFT · 第三次会话（Colab）：卫星场景的正确性检查、显存标定与 HO-FL 对齐（阶段 S）

这次会话检查卫星场景的代码是否正确，测出各分界的显存，并在 AG News 上补两个种子，复核"关掉零阶"带来的差别。预计 1.5 小时左右，预算默认 3 小时。结果汇总在 S6 的反馈报告里，用来判断能否进入后面的实验。

阶段 S 做五件事：

1. **S1 环境、测试、轨迹、数据（基本不用 GPU）**
   - 记录环境；跑一遍不需要 GPU 的测试（原有测试加卫星部分）。
   - 生成 120 颗卫星 24 小时的轨迹，与实现说明附录 A 的数值逐项比对。
   - 下载 HumAID（按事件发布的版本：19 起事件、10 类）和 EuroSAT，生成 3 个种子、两种数据版本的划分，打印统计和哈希。
2. **S2 真实模型上的正确性检查（ViT 和 Qwen）**
   - 显式给出分界时，本地训练的一步与原来的推导方式逐位相同；分界为 0 时梯度与全反向传播一致；分界在最高层时，各层只靠零阶更新。
   - 卫星侧的一次参与，与上游的 `train_client` 逐位相同（Qwen + AG News）。
   - 理想网络下，卫星聚合与"求和后除以人数"的中心化聚合相比：一阶部分逐位相同，零阶部分相差不超过 1e-7。
   - 同一批零阶报告在两个副本上重放，结果逐位相同。
3. **S3 显存标定**：Qwen 在分界 0、8、16、24、28，ViT 在分界 0、3、6、9、12，各做一次完整参与（5 步，批大小 16），记录峰值显存。
4. **S4 冒烟**：本文完整方法在 EuroSAT 和 HumAID 上各跑 2 个仿真小时；另在 EuroSAT 上把理想中心化、只让强星训练、纯零阶三种方法各跑 2 个仿真小时。
5. **S5 与 HO-FL 对齐**：Qwen + AG News，HO-FL 默认设置（β=0.75），零阶开、关，种子 43、44 各一次，共 4 次，每次约 10 分钟。种子 42 用 11-21 那次会话的结果。

前两次会话的实验（C3 到 C8）保留在本 notebook 里，默认关闭。

**步骤一览**

| 步骤 | 内容 | GPU |
|---|---|---|
| A1 到 A3 | 会话、取代码、建环境 | 不用 |
| B | 预演：小模型把基线回归和卫星流程各走一遍，约五分钟 | 不用 |
| C1、C2 | 环境记录；基线回归（默认设置下，结果仍与上游一致） | 很少 |
| S1 | 环境、测试、轨迹、数据与划分 | 不用 |
| S2 | 真实模型上的正确性检查 | 约 10 分钟 |
| S3 | 显存标定 | 约 5 分钟 |
| S4 | 冒烟：5 次运行，各 2 个仿真小时 | 约 10 分钟 |
| S5 | 与 HO-FL 对齐：4 次运行 | 约 40 分钟 |
| S6 | 反馈报告 | 不用 |
| C3 到 C8（默认关） | 第二次会话的实验 | |
| D、E | 汇总、下载结果包、释放 GPU | |

运行时类型选 GPU（默认 G4）。代码只从 GitHub 读取，不使用 Google Drive。`REPO_REF` 默认是 `sat` 分支：**运行前请确认这个分支已经推送到 GitHub**。放不进剩余预算的运行会被跳过，并在报告里列出。

In [ ]:
#@title Parameters { display-mode: "form" }
# The code is read from GitHub only (a public repository: no token, no Colab secrets).
REPO_URL = "https://github.com/GuangLun2000/RHO-FFT.git"  #@param {type:"string"}
# The satellite work lives on the branch "sat"; it must be on GitHub before this notebook can read it.
REPO_REF = "sat"  #@param {type:"string"}

# Total wall-clock budget of this notebook, in hours.
BUDGET_HOURS = 3.0  #@param {type:"number"}

# "venv": Python 3.11 with the pinned dependencies of pyproject.toml (recommended).
# "system": Colab's own Python and torch; only a fallback, versions then differ from the pins.
ENV_MODE = "venv"  #@param ["venv", "system"]
# PyTorch installed only if the pinned 2.6.0 (CUDA 12.4) cannot use the GPU. The G4 runtime is an RTX PRO 6000 of
# the Blackwell generation, which needs a CUDA 12.8 build (PyTorch 2.7 or later).
TORCH_FALLBACK = "2.7.1"  #@param {type:"string"}

# Model and seed of the experiments of session 2 (stages C3-C8). Stage S fixes its own models and seeds.
MODEL = "qwen"  #@param ["qwen", "opt"]
SEED = 42  #@param {type:"integer"}
# Runs at a time. 1 keeps the output of one run together; two runs share a G4 with only about 15% more throughput.
JOBS = 1  #@param {type:"integer"}

# Stages, in the order they run.
RUN_TINY_DRY_RUN = True  #@param {type:"boolean"}
# The CPU test suite runs once, in stage S1; switch this on only if S1 is off.
RUN_TESTS = False  #@param {type:"boolean"}
RUN_REGRESSION = True  #@param {type:"boolean"}
# Stage S: the satellite scenario (correctness checks, memory, smoke runs, alignment with HO-FL).
RUN_SAT_SETUP = True  #@param {type:"boolean"}
RUN_SAT_SELFCHECK = True  #@param {type:"boolean"}
RUN_SAT_MEMORY = True  #@param {type:"boolean"}
RUN_SAT_SMOKE = True  #@param {type:"boolean"}
RUN_SAT_T0 = True  #@param {type:"boolean"}
# The experiments of session 2 (2026-11-22). They have been run; off by default.
RUN_ZO_AG_NEWS = False  #@param {type:"boolean"}
RUN_DOSE_CF = False  #@param {type:"boolean"}
RUN_ZO_CHECK = False  #@param {type:"boolean"}
RUN_E3_PILOT = False  #@param {type:"boolean"}
RUN_ZO_SQUAD = False  #@param {type:"boolean"}
RUN_CONTROLS = False  #@param {type:"boolean"}
# Run the experiments even when the regression check did not pass (not recommended).
FORCE = False  #@param {type:"boolean"}
# Download the result bundle through the browser after every stage and before the runtime is released.
DOWNLOAD_BUNDLE = True  #@param {type:"boolean"}
# Release the GPU runtime after the last cell, and after an error in any cell.
RELEASE_GPU = True  #@param {type:"boolean"}

## A1. 会话与 GPU

In [ ]:
#@title A1 · Session and GPU
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

SESSION_START = time.time()
DEADLINE = SESSION_START + 3600 * BUDGET_HOURS
REPO_DIR = "/content/RHO-FFT"
WORK_P1 = "/content/work/p1"         # results of the P1 pipeline: zeroth-order checks, controls (local disk, fast)
WORK_P2 = "/content/work/p2"         # results of the P2 pipeline: dose-response
WORK_SAT = "/content/work/sat"       # results of the satellite pipeline (stage S)
P0_WORK = "/content/work/p0"         # results of the baseline regression (golden values, verdicts)
TINY = "/content/work/tiny"          # results of the CPU dry run, kept apart from the real ones
BUNDLE = "/content/work/session3_bundle.zip"
CACHE = "/content/cache"             # models, datasets, token caches, traces and partitions
VENV = "/content/venv"
SAT_STAGES = RUN_SAT_SETUP or RUN_SAT_SELFCHECK or RUN_SAT_MEMORY or RUN_SAT_SMOKE or RUN_SAT_T0
OLD_STAGES = RUN_ZO_AG_NEWS or RUN_DOSE_CF or RUN_ZO_CHECK or RUN_E3_PILOT or RUN_ZO_SQUAD or RUN_CONTROLS


def sh(command, cwd=None, env=None, check=True):
    # Run a command, stream its output into the cell, and stop on failure when check is set.
    shown = command if isinstance(command, str) else " ".join(str(part) for part in command)
    print(f"$ {shown}", flush=True)
    process = subprocess.Popen(command, shell=isinstance(command, str), cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {shown}")
    return code


def hours_left():
    return (DEADLINE - time.time()) / 3600


STATUS = {}  # one line per stage, printed in the last cell


def release_gpu(delay=10):
    # Hand the GPU runtime back. Everything on the VM's local disk is lost, so this runs only after the results
    # have been shown and downloaded.
    if not RELEASE_GPU:
        print("RELEASE_GPU is off; the runtime stays connected.")
        return
    print(f"Releasing the GPU runtime in {delay} seconds ...", flush=True)
    time.sleep(delay)
    from google.colab import runtime
    runtime.unassign()


def stop(message):
    # A failed check stops "Run all"; the error handler below then releases the GPU.
    raise RuntimeError(message)


def build_bundle():
    # One zip with the pipelines' bundles (runs, logs, CSV files), their reports and figures. Returns its path or None.
    import zipfile
    parts = [Path(WORK_SAT) / "feedback" / "feedback_E0_bundle.zip", Path(WORK_P1) / "feedback" / "feedback_P1_bundle.zip",
             Path(WORK_P2) / "feedback" / "feedback_P2_bundle.zip"]
    extras = [*Path(WORK_SAT).glob("feedback/*.md"), *Path(WORK_SAT).glob("p0_feedback/*"),
              *Path(WORK_P1).glob("feedback/*.md"), *Path(WORK_P1).glob("feedback/*.png"), *Path(WORK_P2).glob("feedback/*.md"),
              *Path(WORK_P2).glob("feedback/*.png"), *Path(WORK_P1).glob("zo_check/zo_check.*")]
    present = [path for path in parts + extras if path.exists()]
    if not present:
        return None
    with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in present:
            bundle.write(path, f"{path.parent.parent.name}/{path.parent.name}/{path.name}")
    return BUNDLE


def download_results(refresh=True):
    # Send the result bundle to the browser. With refresh, the feedback is rebuilt first from whatever exists, so
    # that an interrupted session still leaves with its finished runs. Returns whether a download was started.
    if refresh and "p1" in globals():
        pipelines = ([("sat", WORK_SAT)] if SAT_STAGES else []) + ([("p1", WORK_P1), ("p2", WORK_P2)] if OLD_STAGES else [])
        for runner, root in pipelines:
            if Path(root).exists():
                try:
                    globals()[runner]("feedback", extra=["--quiet"])
                except Exception as error:
                    print(f"{runner} feedback could not be rebuilt:", repr(error))
    bundle = build_bundle()
    if not (DOWNLOAD_BUNDLE and bundle):
        return False
    try:
        from google.colab import files
        files.download(bundle)
        return True
    except Exception as error:
        print("download failed:", repr(error))
        return False


def _release_on_error(shell, etype, value, traceback, tb_offset=None):
    # Any error in any later cell: show it, save what exists, then release the GPU instead of leaving it idle.
    shell.showtraceback((etype, value, traceback), tb_offset=tb_offset)
    download_results()
    release_gpu(30 if DOWNLOAD_BUNDLE else 10)


# Manual interruption (KeyboardInterrupt) is not an Exception and does not release the runtime.
get_ipython().set_custom_exc((Exception,), _release_on_error)


sh("nvidia-smi", check=False)

## A2. 取代码

仓库是公开的：直接匿名克隆 `REPO_URL` 并检出 `REPO_REF`，不需要 token，也不读取 Colab 的密钥。回归检查要用 `git archive` 取出两个参照提交，所以用完整历史的克隆。

In [ ]:
#@title A2 · Get the code
# The repository is public: an anonymous clone, no credentials and no access to Colab secrets.
git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh(["git", "clone", "--quiet", REPO_URL, REPO_DIR], env=git_env)
else:
    sh(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin"], env=git_env)
remote_ref = f"origin/{REPO_REF}"
has_remote_ref = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", remote_ref],
                                capture_output=True).returncode == 0
sh(["git", "-C", REPO_DIR, "checkout", "--quiet", "--detach", remote_ref if has_remote_ref else REPO_REF])
sh(["git", "-C", REPO_DIR, "log", "--oneline", "-8"])

revisions = json.loads(Path(f"{REPO_DIR}/experiments/p0/revisions.json").read_text())
for name, sha in revisions.items():
    if subprocess.run(["git", "-C", REPO_DIR, "cat-file", "-e", sha + "^{commit}"], capture_output=True).returncode != 0:
        raise RuntimeError(f"Reference revision {name} ({sha}) is missing from this clone; a full-history clone is required.")
print("reference revisions present:", revisions)

## A3. 环境

建一个独立的 Python 3.11 虚拟环境，按 `pyproject.toml` 的固定版本安装依赖（PyTorch 2.6.0 / CUDA 12.4）。所有训练都在这个环境的子进程里运行。仓库本身不安装，运行时通过 `PYTHONPATH` 指向源码树。

G4 运行时是 Blackwell 架构的 RTX PRO 6000，固定版本的 PyTorch 2.6.0（CUDA 12.4）不支持它。环境建好后会先探测 GPU：固定版本用不了时，自动换成参数 `TORCH_FALLBACK` 指定的版本（CUDA 12.8 构建），并在结果里记录实际使用的版本。回归检查在同一环境里进行，所以基线是否被改坏的结论不受影响；与论文数值的对照则多了一层环境差异。H100 上直接使用固定版本。

In [ ]:
#@title A3 · Environment
def clean_environment():
    # Colab exports constraint files for its own package set (PIP_CONSTRAINT, UV_CONSTRAINT, ...). They must not
    # apply to the separate environment built here, which pins different versions.
    return {key: value for key, value in os.environ.items() if not key.startswith(("UV_", "PIP_"))}


if ENV_MODE == "venv":
    uv = f"{sys.executable} -m uv"
    sh(f"{sys.executable} -m pip install -q uv")
    if not os.path.exists(f"{VENV}/bin/python"):
        if sh(f"{uv} venv --python 3.11 {VENV}", env=clean_environment(), check=False) != 0:
            print("Python 3.11 could not be provisioned; building the environment on the runtime's own Python instead")
            shutil.rmtree(VENV, ignore_errors=True)
            sh(f"{uv} venv --python {sys.executable} {VENV}", env=clean_environment())
    # The optional group "sat" adds the land mask (where a satellite is imaging) and Pillow (EuroSAT is stored as JPEG).
    install = f"{uv} pip install --python {VENV}/bin/python -r pyproject.toml --extra dev --extra sat"
    if sh(install + " --torch-backend cu124", cwd=REPO_DIR, env=clean_environment(), check=False) != 0:
        print("retrying without --torch-backend (the default Linux wheel of torch 2.6.0 is the CUDA 12.4 build)")
        sh(install, cwd=REPO_DIR, env=clean_environment())
    PY = f"{VENV}/bin/python"
else:
    PY = sys.executable
    sh(f"{PY} -m pip install -q transformers==4.56.2 peft==0.17.1 datasets==3.6.0 accelerate==1.10.1 "
       "safetensors==0.6.2 sentencepiece==0.2.1 global-land-mask==1.0.0 pillow pytest", env=clean_environment())

sh([PY, "-c", "import torch, transformers, peft, datasets; "
              "print('torch', torch.__version__, '| cuda runtime', torch.version.cuda, '| cuda available', torch.cuda.is_available()); "
              "print('transformers', transformers.__version__, '| peft', peft.__version__, '| datasets', datasets.__version__)"])

# Probe the GPU with the interpreter that will train. The pinned PyTorch has no kernels for Blackwell GPUs (the G4
# runtime); if it cannot use this GPU, the fallback build for CUDA 12.8 is installed and probed again. The versions
# actually used are recorded with every result, and the regression check runs in the same environment.
GPU_PROBE = ("import torch; x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16); "
             "assert torch.isfinite((x @ x).float()).all(); "
             "print('GPU usable:', torch.cuda.get_device_name(0), '| torch', torch.__version__)")


def gpu_usable():
    return sh([PY, "-c", GPU_PROBE], check=False) == 0


if shutil.which("nvidia-smi") is None:
    stop("No GPU in this runtime. Choose a GPU runtime type and run again.")
if not gpu_usable():
    if ENV_MODE != "venv":
        stop("The runtime's own PyTorch cannot use this GPU. Send the output above back.")
    print(f"\nThe pinned PyTorch cannot use this GPU; installing torch {TORCH_FALLBACK} built for CUDA 12.8 instead.")
    fallback = f"{uv} pip install --python {VENV}/bin/python torch=={TORCH_FALLBACK}"
    if sh(fallback + " --torch-backend cu128", env=clean_environment(), check=False) != 0:
        sh(fallback + " --index-url https://download.pytorch.org/whl/cu128", env=clean_environment())
    if not gpu_usable():
        stop("PyTorch cannot use this GPU with either build. Send the output above back.")
    STATUS["environment note"] = f"torch {TORCH_FALLBACK} (CUDA 12.8) in place of the pinned 2.6.0, which does not support this GPU"


def p0(*stages, root=P0_WORK, extra=()):
    # Run stages of experiments/p0/p0.py (baseline regression) with the training interpreter; 0 = all stages ok.
    command = [PY, "experiments/p0/p0.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda", *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def p1(*stages, root=WORK_P1, extra=()):
    # Stages of experiments/p1/p1.py (queue of arms, feedback); 0 = all stages ok.
    command = [PY, "experiments/p1/p1.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda",
               "--model", MODEL, "--seeds", str(SEED), *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def p2(*stages, root=WORK_P2, extra=()):
    # Stages of experiments/p2/p2.py (base run, counterfactual dose-response, feedback); 0 = all stages ok.
    command = [PY, "experiments/p2/p2.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda",
               "--model", MODEL, "--seed", str(SEED), *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def sat(*stages, root=WORK_SAT, extra=()):
    # Stages of experiments/sat/e0/e0.py (satellite scenario: checks, memory, smoke runs, HO-FL alignment); 0 = all ok.
    command = [PY, "experiments/sat/e0/e0.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda", *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def stage(title, text=""):
    # A Markdown heading in the output: a copy of the page into a .md file gets one section per stage.
    print(f"\n# {title}\n" + (f"\n{text}\n" if text else ""), flush=True)


def print_report(which, sections=None):
    # Rebuild a feedback report from the runs finished so far and print it as Markdown text (it copies cleanly).
    runner = {"p1": p1, "p2": p2}[which]
    if sections:
        runner("feedback", extra=["--sections", sections])
        return
    runner("feedback", extra=["--quiet"])
    path = Path({"p1": WORK_P1, "p2": WORK_P2}[which]) / "feedback" / f"feedback_{which.upper()}.md"
    if path.exists():
        print(path.read_text(), flush=True)


def save_bundle():
    started = download_results(refresh=False)
    print("result bundle:", "download started" if started else "not downloaded")


def allowed(label):
    # Whether a GPU stage may start: the regression must have passed (or FORCE) and some budget must be left.
    if regression_ok is False and not FORCE:
        print(f"{label} skipped: the regression check did not pass. Set FORCE = True to run it anyway.")
        STATUS[label] = "skipped (regression did not pass)"
        return False
    if hours_left() < 0.3:
        print(f"{label} skipped: less than 18 minutes of budget left.")
        STATUS[label] = "skipped (no budget left)"
        return False
    if regression_ok is None:
        print("NOTE: the regression check was not run in this session; the results build on an unverified baseline.")
    return True


def finish_stage(label, done, which, sections):
    STATUS[label] = "finished" if done else "not finished (see the queue table above)"
    print(f"\n{label}: {STATUS[label]}; {hours_left():.2f} h of budget left\n", flush=True)
    if sections:
        print(f"## What this stage shows (report sections {sections}; the full report follows at the end)\n", flush=True)
        print_report(which, sections)
    save_bundle()


def run_p1_block(block, label, sections):
    # One block of arms.json under the remaining budget (ten minutes are kept for feedback and download).
    if not allowed(label):
        return False
    done = p1("queue", extra=["--blocks", str(block), "--jobs", str(JOBS), "--deadline", str(DEADLINE - 600)]) == 0
    finish_stage(label, done, "p1", sections)
    return done


def run_sat_stage(stages, label, budgeted=False):
    # Stages of the satellite pipeline. Stages with long runs get the session deadline and skip what does not fit.
    if not allowed(label):
        return False
    done = sat(*stages, extra=["--deadline", str(DEADLINE - 600)] if budgeted else []) == 0
    # Rebuild the report and its bundle from what exists, so that the download after this stage holds it.
    sat("feedback", extra=["--quiet"])
    finish_stage(label, done, "sat", None)
    return done


def run_p2_stage(name, label, sections=None):
    if not allowed(label):
        return False
    done = p2(name, extra=["--jobs", str(JOBS), "--deadline", str(DEADLINE - 600)]) == 0
    finish_stage(label, done, "p2", sections)
    return done

## B. 预演（CPU，约五分钟）

用随机初始化的小模型和合成数据，把基线回归和卫星流程（轨迹、数据划分、正确性检查、冒烟、与 HO-FL 对齐、反馈报告）各走一遍。只检查流程能不能跑通，数字没有意义，所以每一步只打印一行结果；失败时打印它的最后 60 行，notebook 停下并释放 GPU。第二次会话的实验打开时，它们的预演也会运行。

In [ ]:
#@title B · Dry run on CPU
def dry(label, command):
    # One dry-run step with its output kept out of the page; its end is shown only when it fails.
    started = time.time()
    result = subprocess.run(command, cwd=REPO_DIR, capture_output=True, text=True)
    ok = result.returncode == 0
    STATUS[f"dry run: {label}"] = "ok" if ok else "FAILED"
    print(f"- {label}: {'ok' if ok else 'FAILED'} ({time.time() - started:.0f} s)", flush=True)
    if not ok:
        print("\n".join((result.stdout + result.stderr).splitlines()[-60:]))
        stop(f"The dry run of the {label} failed. Nothing has been run on the GPU yet; send the output above back.")


if RUN_TINY_DRY_RUN:
    stage("B · Dry run on CPU", "Tiny random model and synthetic data: checks that every pipeline of this session runs; its numbers mean nothing.")
    dry("baseline regression (P0)", [PY, "experiments/p0/p0.py", "env", "golden", "regress", "--tiny", "--root", f"{TINY}/p0"])
    if SAT_STAGES:
        dry("satellite pipeline (trace, partitions, self-checks, memory, smoke runs, HO-FL alignment, report)",
            [PY, "experiments/sat/e0/e0.py", "all", "--tiny", "--quiet", "--root", f"{TINY}/sat"])
    if OLD_STAGES:
        dry("dose-response (P2: base run, counterfactual trajectory, report)", [PY, "experiments/p2/p2.py", "all", "--tiny", "--quiet", "--jobs", "2", "--root", f"{TINY}/p2"])
        tiny_state = next(Path(TINY).glob("p2/runs/*dose-base/states/*.safetensors"))
        dry("estimator check", [PY, "-m", "ho_fl.zo_check", "--tiny", "--states", str(tiny_state), "--q", "16", "--batches", "1", "--output", f"{TINY}/p1/zo_check"])
        dry("P1 arms of blocks 5, 7, 4 and the report", [PY, "experiments/p1/p1.py", "queue", "feedback", "--tiny", "--quiet", "--jobs", "2",
                                                         "--blocks", "5,7,4", "--root", f"{TINY}/p1"])
    print(f"\ndry run ok, {hours_left():.2f} h of budget left")

## C1. 环境记录

记录 GPU 型号、驱动和各依赖的版本，并做一次 BF16 探测。不需要 GPU 的测试在 S1 里跑一遍（参数 `RUN_TESTS` 打开时，这里也会跑）。

In [ ]:
#@title C1 · Environment record and unit tests
stage("C1 · Environment record and unit tests")
env_ok = p0("env") == 0
STATUS["environment"] = "ok" if env_ok else "FAILED"
if not env_ok:
    stop("Environment check failed (no CUDA device, bfloat16 not usable, or a reference revision is missing). See the warnings above.")
if RUN_TESTS:
    tests_ok = p0("tests") == 0
    STATUS["unit tests"] = "ok" if tests_ok else "FAILURES"
    print("unit tests:", "ok" if tests_ok else "FAILURES (listed above)")

## C2. 基线回归

三个冒烟配置（HO-FL、FedAvg、DeComFL，OPT-125M / SST-2，3 轮）在上游原样提交上各跑一次，在插桩提交上各跑两次，在当前代码上各跑一次，再加两次 `ho-fl-replay`。卫星部分改动了 `hybrid_step`（可以显式给出分界）、`local_step` 和层号的识别，这一步检查默认配置下的结果仍与上游一致。

读法：`I1 verdict` 为 `PASS_BITWISE` 或 `PASS_TOLERANCE` 即通过；`FAIL` 说明当前代码改变了基线，后面的阶段默认不再运行；`INCONCLUSIVE` 说明这块 GPU 上参照提交自己两次运行的差就超过了容差，需要研究负责人决定怎么判。

In [ ]:
#@title C2 · Golden values and regression (I1, I2)
regression_ok = None
if RUN_REGRESSION:
    stage("C2 · Golden values and baseline regression (I1, I2)", "Three smoke configurations (HO-FL, FedAvg, DeComFL on OPT-125M / SST-2, 3 rounds) on the unchanged upstream commit and on this code: I1 passes when the results are identical, I2 when a replay reproduces them.")
    golden_ok = p0("golden") == 0
    regression_ok = golden_ok and p0("regress") == 0
    STATUS["golden values"] = "ok" if golden_ok else "ATTENTION"
    STATUS["regression (I1, I2)"] = "ok" if regression_ok else "ATTENTION"
    p0("feedback", extra=["--quiet"])
    # The regression record travels with the bundles of the pipelines that run.
    for work in ([WORK_SAT] if SAT_STAGES else []) + ([WORK_P1, WORK_P2] if OLD_STAGES else []):
        target = Path(work) / "p0_feedback"
        target.mkdir(parents=True, exist_ok=True)
        for path in [*Path(P0_WORK).glob("feedback/feedback_P0.*"), *Path(P0_WORK).glob("golden/*/golden.json"),
                     *Path(P0_WORK).glob("regress/*/verdicts.json"), Path(P0_WORK) / "env.json"]:
            if path.exists():
                shutil.copy2(path, target / (path.parent.name + "__" + path.name if path.name in ("golden.json", "verdicts.json") else path.name))
    print("\ngolden:", "ok" if golden_ok else "ATTENTION", "| regression (I1, I2):", "ok" if regression_ok else "ATTENTION")
    print(f"{hours_left():.2f} h of budget left")

## 本次会话的计划

打印本次会话要跑的每个阶段：跑哪些运行、回答什么问题、大约多久、这次是否打开。复制输出时，这张表就是这份记录的目录。

In [ ]:
#@title Plan · What this session runs
PLAN = [
    ("S1", RUN_SAT_SETUP, "Satellite: environment, tests, trace, data", "no training",
     "Do all tests pass; does the trace agree with appendix A of the brief; the partitions and their hashes", "about 10 min, no GPU"),
    ("S2", RUN_SAT_SELFCHECK, "Satellite: correctness on the real models", "no training runs",
     "Explicit cuts, one participation against upstream, aggregation against the central rule, replay on two replicas", "about 10 min"),
    ("S3", RUN_SAT_MEMORY, "Satellite: memory calibration", "one participation per cut, Qwen and ViT",
     "Peak GPU memory per cut: what an idle and an imaging GPU satellite, and a forward-only satellite, need", "about 5 min"),
    ("S4", RUN_SAT_SMOKE, "Satellite: smoke runs", "full method on EuroSAT and HumAID; ideal, backbone-only and zeroth-order on EuroSAT (2 simulated hours each)",
     "Does a run work end to end; seconds per round and per evaluation for the budget of the experiments", "about 10 min"),
    ("S5", RUN_SAT_T0, "HO-FL with and without its zeroth-order part", "Qwen / AG News, seeds 43 and 44, zeroth-order on and off",
     "Does removing the zeroth-order part cost accuracy (authors: 3.32 points; seed 42 on 2026-11-21: 0.04)", "about 40 min"),
    ("C3", RUN_ZO_AG_NEWS, "Zeroth-order check, AG News", "A-b0.75-attr, Au24-b0, Bu24-b0",
     "Do the zeroth-order components lower the loss (attribution)? Does session 1 reproduce exactly? Paper Table 9 (63.22) with zeroth-order on and off", "about 50 min"),
    ("C4", RUN_DOSE_CF, "E2 dose-response (counterfactual)", "dose-base (40 rounds), dose-cf-m2 (20 rounds)",
     "H2: from how many honest first-order reports does first-order-only robust averaging win; which fallback below the quorum", "about 15 min"),
    ("C5", RUN_ZO_CHECK, "Zeroth-order estimator check", "no training",
     "Is HO-FL's finite difference accurate in bfloat16; how much does a q = 2 estimate say about the gradient", "about 10 min"),
    ("C6", RUN_E3_PILOT, "E3 pilot", "B2-lr20, B2-label",
     "Is the damage of one faulty device large enough for E3 to separate configurations", "about 35 min"),
    ("C7", RUN_ZO_SQUAD, "Top-FO-Only on SQuAD", "A-b0.75-squad, B-b0.75-squad", "The rebuttal's SQuAD numbers (F1 86.92 to 83.65)", "about 1.5 h"),
    ("C8", RUN_CONTROLS, "Controls", "Cx2-b0, fedavg-x2, C-b0-r480, A-b0-r480", "Step size and training length behind the fo_mean gain (H1a)", "about 1.2 h"),
]
stage("Plan of this session", f"Budget {BUDGET_HOURS} h ({hours_left():.2f} h left). Stage S is the satellite scenario; "
      f"stages C3 to C8 are the experiments of session 2 (model {MODEL}, seed {SEED}), which have been run and are off by default.")
print("| stage | runs | question | time | this session |\n| --- | --- | --- | --- | --- |")
for key, enabled, title, runs, question, duration in PLAN:
    print(f"| {key} {title} | {runs} | {question} | {duration} | {'yes' if enabled else 'no (switched off)'} |")

## C3. 零阶检验，AG News（3 次运行，约 50 分钟）

- `A-b0.75-attr`：HO-FL 默认设置（β=0.75），加归因探针。每 4 轮在 1,024 条固定的评估样本上求一次梯度，把这一轮实际施加的更新带来的（一阶近似的）损失下降，拆成一阶分量和零阶分量各自的贡献，按层段汇总。探针和评估都只读，训练与第一次会话的 `A-b0.75` 完全相同：报告在共同的评估点（每 20 轮）逐个比对，相同则两次会话的结果可以配对。它每 4 轮评估一次并记录各类准确率（E3 的要求），所以也是 C6 两个故障运行的无故障对照。
- `Au24-b0`：所有设备都用最高的零阶分界（第 24 层，只有最上面 4 层做一阶），β=0，复现论文表 9 的 63.22；同样加归因探针。
- `Bu24-b0`：同一设置下冻结底层（零阶关闭，直接平均），即这个设置下的 Top-FO-Only。

AG News 上默认设置的 Top-FO-Only（作者 83.37 对 86.69）用第一次会话的 `B-b0.75` 和 `A-b0.75`（85.95 对 85.99），不重跑。

In [ ]:
#@title C3 · Zeroth-order check on AG News (block 5)
if RUN_ZO_AG_NEWS:
    stage("C3 · Zeroth-order check on AG News",
          "Does switching the zeroth-order updates off change anything, measured three ways: the attribution probe (which part of each "
          "round's loss decrease comes from zeroth-order components), the paper's uniform highest boundary with zeroth-order on and off, "
          "and the cross-session check of A-b0.75-attr against session 1. While a run trains, the output shows every evaluation and one "
          "line per attribution round; after each run, its result and its comparison with the runs it is read against.")
    run_p1_block(5, "C3 zeroth-order check, AG News", "2")
else:
    print("C3 · Zeroth-order check on AG News: switched off")

## C4. 剂量响应：起点运行与反事实轨迹（约 15 分钟）

先用默认设置跑 40 轮，保存第 40 轮的状态作为起点；再从这个状态跑一条 20 轮的轨迹（名单按剂量 m=2 的布局：指定的那台 T1 设备加上另外 5 台）。每一轮，另外 5 台设备都在另一个分界上用同样的批次再训练一次，于是同一轮里可以组合出 m=0 到 5 的所有剂量；再对每个剂量用 8 种聚合规则、3 种故障强度（无、取反、取反并放大 20 倍）各算一次聚合结果，记录故障的影响和损失的下降。探针只读，不改变这条轨迹。输出里每一轮有一段：零阶报告相对一阶报告的贡献，以及 ×20 故障下几种规则在各剂量的进展。

In [ ]:
#@title C4 · Dose-response: base run and counterfactual trajectory
if RUN_DOSE_CF:
    stage("C4 · E2 dose-response (counterfactual)",
          "H2 and the fallback rule of RHO-FFT: with one unreliable first-order device on the scarce segment S1 and m honest "
          "first-order reports beside it, which aggregation keeps learning and limits the fault, for every m from 0 to 5.")
    if run_p2_stage("base", "C4 E2 base run"):
        run_p2_stage("cf", "C4 E2 counterfactual trajectory", "3,7")
    else:
        STATUS["C4 E2 counterfactual trajectory"] = "skipped (no base run)"
else:
    print("C4 · E2 dose-response: switched off")

## C5. 零阶估计器的数值检查（不训练，约 10 分钟）

在初始状态和上一步保存的第 40 轮状态上，对三档设备各取 2 个批次、64 个方向：调用训练用的 `hybrid_step` 得到 HO-FL 的标量 v，再用 autograd 算出同一方向上的精确方向导数，比较 v/μ 与它（μ 取 1e-4、1e-3、1e-2）。先在 BF16 主干上做（与训练相同），再把主干转成 FP32 做一遍作为参照。另外给出 q 个方向的估计与真实梯度的余弦随 q 的变化，以及理论值 √(q/d)。

In [ ]:
#@title C5 · Numerical check of the zeroth-order estimator
if RUN_ZO_CHECK and allowed("C5 estimator check"):
    stage("C5 · Zeroth-order estimator check (no training)",
          "HO-FL's scalar v / mu against the exact directional derivative from autograd, on the directions training draws, at the "
          "initial state and at the round-40 state of C4; bfloat16 backbone as in training, then float32 as the reference.")
    base_state = Path(WORK_P2) / "runs" / f"{MODEL}-s{SEED}-dose-base" / "states" / "round_0040.safetensors"
    states = str(base_state) if base_state.exists() else ""
    code = sh([PY, "-m", "ho_fl.zo_check", "--states", states, "--output", f"{WORK_P1}/zo_check", "--model", MODEL, "--dataset", "ag_news",
               "--partition", "dirichlet_05", "--seed", str(SEED), "--device", "cuda", "--cache-dir", CACHE], cwd=REPO_DIR, check=False)
    STATUS["C5 estimator check"] = "finished" if code == 0 else "FAILED (see above)"
    print(f"\nC5 estimator check: {STATUS['C5 estimator check']}; {hours_left():.2f} h of budget left")
elif not RUN_ZO_CHECK:
    print("C5 · Zeroth-order estimator check: switched off")

## C6. E3 预试：一台故障设备（2 次运行，约 35 分钟）

E3 在 Qwen 上的两个格子：HO-FL 默认设置（B2：`dep_round_v1`，β=0.75，等权平均），偏向集合里排名最前的那台 T1 设备（约 80% 的轮次在场）在第 41 到 80 轮出故障。160 轮，每 4 轮评估一次，记录各类准确率，发散时停下并记为结果。

- `B2-lr20`：这台设备的本地学习率 ×20（配置错误）；
- `B2-label`：这台设备的全部标签换成下一类（标注流程出错）。

对照是 C3 的 `A-b0.75-attr`：同一种子、同一数据流、前 40 轮完全相同，所以之后的差距全部来自故障。每个运行结束后立即打印说明 §6.11 的韧性指标：窗口内平均下降、最大下降、恢复轮数、最终差距。最大下降不到 1 个点（约两倍种子标准差）时，E3 用这种故障区分不了配置，要先加强故障或改窗口，再写 E3 的实验网格。

In [ ]:
#@title C6 · E3 pilot: one faulty device (block 7)
if RUN_E3_PILOT:
    stage("C6 · E3 pilot: one faulty device under HO-FL's default",
          "The best-ranked T1 device of the preferred set (in about 80% of the cohorts) is faulty in rounds 41-80: local learning rate x20 "
          "(B2-lr20) or every label shifted to the next class (B2-label). Each is read against A-b0.75-attr of C3, which trains identically "
          "without the fault, so the drop after round 40 is the effect of the fault (before round 41 it must be exactly 0).")
    run_p1_block(7, "C6 E3 pilot", "2c")
else:
    print("C6 · E3 pilot: switched off")

## C7. 零阶检验，SQuAD：Top-FO-Only（2 次运行，默认关，约 1.5 小时）

作者在 rebuttal 里报告 SQuAD 上冻结底层后 F1 从 86.92 降到 83.65。这里按同样的联邦设置跑 HO-FL（β=0.75）和冻结底层两次，只在最后做一次完整验证集的生成式评估（EM/F1），不做中间评估。先跑一次 3 轮的预检来测耗时；放不进剩余预算会被跳过。

In [ ]:
#@title C7 · Zeroth-order check on SQuAD (block 6)
if RUN_ZO_SQUAD:
    stage("C7 · Top-FO-Only on SQuAD", "HO-FL (beta 0.75) and the same with the bottom frozen; one generative evaluation on the full validation set at the end.")
    run_p1_block(6, "C7 Top-FO-Only, SQuAD", "2")
else:
    print("C7 · Top-FO-Only on SQuAD: switched off")

## C8. 对照（4 次运行，默认关，约 1.2 小时）

`Cx2-b0`（`fo_mean` 步长 ×2，对比第一次会话的 `Ax2-b0`：步长相同时只让一阶报告投票是否仍更好）、`fedavg-x2`（上限本身是否欠步长）、`C-b0-r480` 和 `A-b0-r480`（训练三倍长，收敛后差距是否还在）。这些运行偏离"不单独调参"的规则，只作诊断。

In [ ]:
#@title C8 · Controls (block 4)
if RUN_CONTROLS:
    stage("C8 · Controls (diagnostics, not proposed methods)", "Larger server steps and three times as many rounds, read against the session-1 runs.")
    run_p1_block(4, "C8 controls", "2b,3")
else:
    print("C8 · Controls: switched off")

## S. 卫星场景：正确性检查、显存标定与 HO-FL 对齐

下面六步都由 `experiments/sat/e0/e0.py` 完成，结果在 `/content/work/sat` 下。每一步都可以重跑：已完成的部分会直接复用。

### S1. 环境、测试、轨迹、数据（基本不用 GPU）

- 测试：原有的 CPU 测试和 `tests/sat/` 下的新测试一起跑。
- 轨迹：输出一张表，逐项对比轨迹统计与附录 A 的数值（周期、星间距离、地影比例、成像比例、地面站可见比例、可达轨道面数），并给出轨迹哈希。
- 数据：打印 HumAID 和 EuroSAT 在种子 42 下的划分统计（每个轨道面的类别构成、主事件占比），以及 3 个种子、两种数据版本共 12 份划分的哈希，并与里程碑报告里记录的哈希比对。

In [ ]:
#@title S1 · Satellite: environment, tests, trace, data
if RUN_SAT_SETUP:
    stage("S1 · Satellite scenario: environment, tests, trace, data",
          "The CPU test suite (upstream tests and tests/sat); the 24-hour trace of 120 satellites against appendix A of the brief; "
          "HumAID and EuroSAT with the partitions of seeds 42, 43, 44 in both data variants.")
    run_sat_stage(["env", "tests", "trace", "data"], "S1 satellite setup")
else:
    print("S1 · Satellite setup: switched off")

### S2. 真实模型上的正确性检查

在 ViT-B/16（EuroSAT）和 Qwen2.5-1.5B（HumAID）上各跑一遍，另用 Qwen + AG News 与上游的训练函数比对。每行是一项检查，`pass` 为通过；任何一行是 `FAIL`，后面的实验都不该做，请把输出带回来。表里的 `bitwise True / False` 说明在这块 GPU 上结果是否逐位相同（零阶部分的要求是相差不超过 1e-7）。

In [ ]:
#@title S2 · Satellite: correctness checks on the real models
if RUN_SAT_SELFCHECK:
    stage("S2 · Correctness checks on the real models",
          "Explicit cuts against the derived ones, a participation against upstream train_client, the satellite aggregation "
          "against the central sum divided by the number of participants, and the replay of zeroth-order reports on two replicas.")
    run_sat_stage(["selfcheck"], "S2 satellite self-check")
else:
    print("S2 · Satellite self-check: switched off")

### S3. 显存标定

对每个分界做一次完整参与（5 步本地训练，批大小 16；Qwen 的输入长度取 192，ViT 的输入是 224×224），记录峰值显存。表里另有两组参照：只做一次前向推理，以及冻结分界以下的层（不做零阶）。场景里"GPU 卫星空闲 / 成像"两种状态的显存，分别取分界 8 / 16（ViT 为 3 / 6）的峰值。

In [ ]:
#@title S3 · Satellite: peak GPU memory per cut
if RUN_SAT_MEMORY:
    stage("S3 · Peak GPU memory of a participation per cut", "Qwen2.5-1.5B at cuts 0, 8, 16, 24, 28 and ViT-B/16 at cuts 0, 3, 6, 9, 12; batch 16.")
    run_sat_stage(["memory"], "S3 memory calibration")
else:
    print("S3 · Memory calibration: switched off")

### S4. 冒烟（5 次运行，各 2 个仿真小时）

一个仿真小时是 6 轮，每轮约 18 颗卫星参与。运行时每个仿真小时打印一行训练损失和一行测试精度。这一步看三件事：流程能否跑完、损失是否下降且没有非有限值、每轮和每次评估各花多少秒（用来估算后面实验的 GPU 小时）。

- 本文完整方法：EuroSAT、HumAID 各一次；
- 理想中心化、只让强星训练、纯零阶：EuroSAT 各一次。

服务器步长都取 1，没有调过，所以这里的精度不能用来比较方法。

In [ ]:
#@title S4 · Satellite: smoke runs
if RUN_SAT_SMOKE:
    stage("S4 · Smoke runs: two simulated hours each",
          "The full method on EuroSAT and on HumAID; the ideal centralised, backbone-only and zeroth-order methods on EuroSAT. "
          "Server step 1 everywhere (not tuned): these accuracies do not compare methods.")
    run_sat_stage(["smoke"], "S4 smoke runs")
else:
    print("S4 · Smoke runs: switched off")

### S5. 与 HO-FL 对齐：零阶开和关（4 次运行，约 40 分钟）

与 11-22 状态报告相同的设置：Qwen2.5-1.5B、AG News（Dirichlet 0.5）、HO-FL 默认采样（β=0.75）、160 轮。种子 43、44 各跑两次：零阶打开，以及关掉零阶（冻结分界以下的层）。种子 42 用 11-21 会话的结果（85.99 对 85.95）。S6 的报告给出三个种子上"开 − 关"的差值、均值和标准差；作者报告的差值是 3.32 个点。放不进剩余预算的运行会被跳过。

In [ ]:
#@title S5 · HO-FL with and without its zeroth-order part (seeds 43, 44)
if RUN_SAT_T0:
    stage("S5 · HO-FL on AG News with the zeroth-order part on and off",
          "Qwen2.5-1.5B, Dirichlet 0.5, beta 0.75, 160 rounds; seeds 43 and 44. Seed 42 is read from the session of 2026-11-21.")
    run_sat_stage(["t0"], "S5 HO-FL alignment", budgeted=True)
else:
    print("S5 · HO-FL alignment: switched off")

### S6. 反馈报告

把上面各步汇总成一份报告并打印：正确性检查的清单（七项，逐项给出通过、不通过或未运行）、轨迹与划分的哈希、两张显存表、冒烟运行的精度与耗时、零阶开关在三个种子上的差值。

In [ ]:
#@title S6 · Satellite: feedback report
if SAT_STAGES:
    stage("S6 · Feedback report of stage S", "Rebuilt from everything stage S has finished so far.")
    code = sat("feedback")
    STATUS["S6 satellite report"] = "written, every check that ran passed" if code == 0 else "written, ATTENTION: a check failed (see the checklist)"
    save_bundle()
else:
    print("S6 · Satellite report: stage S is switched off")

## D. 第二次会话实验的完整报告与图（这些实验打开时）

两份完整报告以 Markdown 文本打印（复制到 .md 文件后表格保持原样）：P1 报告的第 2 节是零阶检验，第 2c 节是 E3 预试；P2 报告的第 3 节是反事实剂量响应。之后显示图。卫星部分的报告在 S6。

In [ ]:
#@title D · Full reports and figures
from IPython.display import Image, display

if OLD_STAGES:
    stage("D · Full reports", "Rebuilt from every run of this session; the stage sections above are excerpts of these.")
    print_report("p1")
    print_report("p2")
    STATUS["feedback"] = "written"
else:
    print("D · The experiments of session 2 are switched off; the report of stage S is printed in S6.")


def read_feedback(root, name):
    path = Path(root) / "feedback" / name
    return json.loads(path.read_text()) if path.exists() else {}


data_p1, data_p2 = read_feedback(WORK_P1, "feedback_P1.json"), read_feedback(WORK_P2, "feedback_P2.json")
# The figures are drawn by the notebook's own Python, which has matplotlib.
for script, root, name in (("p1/plot_curves.py", WORK_P1, "feedback_P1.json"), ("p2/plot_dose.py", WORK_P2, "feedback_P2.json")):
    if (Path(root) / "feedback" / name).exists():
        sh([sys.executable, f"{REPO_DIR}/experiments/{script}", f"{root}/feedback/{name}", f"{root}/feedback"], check=False)
for path in [Path(WORK_P1) / "feedback" / name for name in ("p1_zo_check.png", "p1_e3_pilot.png", "p1_curves.png", "p1_controls.png")] + [Path(WORK_P2) / "feedback" / "p2_dose_cf.png"]:
    if path.exists():
        print(f"\nFigure: {path.name}")
        display(Image(filename=str(path)))

## E. 汇总并释放 GPU

以 Markdown 表格打印各阶段的状态和主要判定，让浏览器下载结果包，然后释放 GPU 运行时。

In [ ]:
#@title E · Summary, download, then release the GPU
data_p1, data_p2 = globals().get("data_p1") or {}, globals().get("data_p2") or {}
stage("E · Session summary", f"Time used: {(time.time() - SESSION_START) / 3600:.2f} h of {BUDGET_HOURS} h.")
print("| stage | status |\n| --- | --- |")
for name, state in STATUS.items():
    print(f"| {name} | {state} |")
sat_report = Path(WORK_SAT) / "feedback" / "feedback_E0.json"
if sat_report.exists():
    from importlib import util as _util
    _spec = _util.spec_from_file_location("sat_e0", f"{REPO_DIR}/experiments/sat/e0/e0.py")
    _e0 = _util.module_from_spec(_spec)
    _spec.loader.exec_module(_e0)
    print("\n| satellite check | verdict | evidence |\n| --- | --- | --- |")
    for item, what, verdict, evidence in _e0.checklist(json.loads(sat_report.read_text())):
        print(f"| {item}. {what} | {verdict} | {evidence} |")
show = lambda value: "n/a" if value is None else f"{value:.2f}"  # noqa: E731
if data_p1.get("statements"):
    print("\n| HO-FL authors' statement | authors: on / off | ours: on / off | verdict |\n| --- | --- | --- | --- |")
    for s in data_p1["statements"]:
        print(f"| {s['source']}; {s['task']} | {show(s['authors_on'])} / {show(s['authors_off'])} | {show(s['ours_on'])} / {show(s['ours_off'])} | {s['verdict']} |")
print()
canary = data_p1.get("canary")
if canary:
    print("- Cross-session check (A-b0.75-attr against A-b0.75 of session 1): "
          + ("identical, so the sessions can be paired" if canary["identical"] else f"DIFFERENT at rounds {canary['differing_rounds'][:8]}"))
for entry in (data_p1.get("pilot") or {}).get("arms", []):
    metrics = entry["metrics"] or {}
    print(f"- E3 pilot {entry['arm']}: {entry['status']}; mean drop in the window {show(metrics.get('mean_drop'))}, largest {show(metrics.get('max_drop'))}, "
          f"final gap {show(metrics.get('final_gap'))} points")
if data_p2.get("cf"):
    print("- H2 (counterfactual): " + ", ".join(f"{fault} {entry['verdict']}" for fault, entry in data_p2["cf"]["h2"].items()))
started = download_results(refresh=False)  # the reports were rebuilt in D
print("\nresult bundle:", "download started" if started else "NOT downloaded; the reports above are the record of this session")
release_gpu(30 if started else 10)

## 带回代码库的内容

- 整页输出：复制到 `colab_results_feedback/` 下新建的目录里（例如 `v3_<日期>/`）的 .md 文件。每个阶段以 `# S2 · ...` 这样的标题开始，表格是 Markdown 文本，复制后保持原样。
- 浏览器下载的 **`session3_bundle.zip`**：卫星部分的报告 `feedback_E0.md` 和结果包（环境记录、测试结果、轨迹的元数据与统计、12 份划分的哈希、正确性检查的逐项结果、两张显存表、每次运行的 `config.json`、`summary.json`、`rounds.jsonl`、`ledger.jsonl`、`eval.jsonl` 和日志），以及基线回归的记录。第二次会话的实验打开时，它们的报告和结果包也在里面。